# Cybersecurity — C2 Beaconing Detection (Jittered Intervals)

Simulate network beacons with jitter and missing packets.
Use inter-arrival autocorrelation / FFT to estimate the beacon interval and compare to QFT peaks for intuition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, rfftfreq
from quantum_hybrid_system import PeriodicState

rng = np.random.default_rng(5)
T = 10000
interval = 120  # seconds, nominal
jitter = 15
timeline = np.zeros(T)
t = 0
while t < T:
    if rng.random() > 0.1:  # 10% packet loss
        timeline[t] = 1.0
    t += max(1, int(interval + rng.integers(-jitter, jitter+1)))

plt.figure()
plt.plot(timeline[:1000])
plt.title("Beacon timeline (first 1000 seconds)")
plt.xlabel("t (s)")
plt.ylabel("event")
plt.show()

yf = np.abs(rfft(timeline))
xf = rfftfreq(T, d=1.0)
k = np.argmax(yf[1:]) + 1
freq = xf[k]
period_est = 1/freq
print("Estimated beacon interval ~", period_est, "s")

n = 10
r = max(2, min(int(round(period_est)), 128))
ps = PeriodicState(num_qubits=n, period=r)
samples = ps.measure(num_shots=4000, use_qft=True)

bins = 64
hist = np.zeros(bins, dtype=int)
N = 2**n
for s in samples:
    hist[(s * bins) // N] += 1

plt.figure()
plt.bar(np.arange(bins), hist)
plt.title(f"QFT histogram (interval r≈{r} samples)")
plt.xlabel("coarse bin")
plt.ylabel("counts")
plt.show()